In [1]:
from google.colab import files, auth
from google.auth import default
from googleapiclient.discovery import build
from pathlib import Path
import io, zipfile, csv, collections, time
import numpy as np
from PIL import Image
from scipy.ndimage import distance_transform_edt
from scipy.spatial import cKDTree

# From notebook 1 MMA3001 apple tree dataset
dataset_zip = Path('/content/MMA3001_appletree_dataset.zip')

print('Upload MMA3001_appletree_dataset.zip')

files.upload()

with zipfile.ZipFile(dataset_zip) as z:
    assert 'inventory.csv' in z.namelist(), 'inventory.csv is missing.'


auth.authenticate_user()
credentials, _ = default()
service = build('drive', 'v3', credentials=credentials)



Upload MMA3001_appletree_dataset.zip


Saving MMA3001_appletree_dataset.zip to MMA3001_appletree_dataset.zip


In [2]:

# Split depth pixels into training, validation and testing
def make_split(depth, seed=42, block_size=None):
    valid = depth > 0

    if block_size is None:
        positions = np.argwhere(valid)
    else:
        h, w = depth.shape
        positions = [
            (y, x)
            for y in range(0, h - block_size + 1, block_size)
            for x in range(0, w - block_size + 1, block_size)
            if valid[y:y+block_size, x:x+block_size].mean() >= 0.5
        ]
        positions = np.array(positions)

    order = np.random.default_rng(seed).permutation(len(positions))
    n = max(1, int(len(positions) * 0.05))

    val = np.zeros(depth.shape, dtype=bool)
    test = np.zeros(depth.shape, dtype=bool)

    for mask, chosen in [(val, order[:n]), (test, order[n:2*n])]:
        for i in chosen:
            y, x = positions[i]
            if block_size is None:
                mask[y, x] = True
            else:
                mask[y:y+block_size, x:x+block_size] = True

    val &= valid
    test &= valid

    return valid & ~val & ~test, val, test


def make_block_split(depth, block_size=12, seed=42):
    return make_split(depth, seed, block_size)


# Nearest-neighbour or IDW interpolation
def interpolate_depth(depth, train, query, method='nearest', radius=6):
    prediction = np.full(depth.shape, np.nan)
    supported = np.zeros(depth.shape, dtype=bool)

    if not train.any() or not query.any():
        return prediction, supported

    if method == 'nearest':
        distances, indices = distance_transform_edt(
            ~train, return_indices=True
        )
        supported = query & (distances <= radius)
        prediction[supported] = depth[
            tuple(index[supported] for index in indices)
        ]

    elif method == 'idw':
        known = np.argwhere(train)
        targets = np.argwhere(query)

        k = min(8, len(known))
        distances, indices = cKDTree(known).query(
            targets, k=k, distance_upper_bound=radius
        )

        if k == 1:
            distances = distances[:, None]
            indices = indices[:, None]

        ok = np.isfinite(distances) & (indices < len(known))
        weights = np.zeros_like(distances)
        weights[ok] = 1 / np.maximum(distances[ok], 1e-10)**2

        nearby = np.zeros_like(weights)
        nearby[ok] = depth[train][indices[ok]]

        total = weights.sum(axis=1)
        found = total > 0
        y, x = targets[found].T

        prediction[y, x] = (
            (nearby[found] * weights[found]).sum(axis=1)
            / total[found]
        )
        supported[y, x] = True

    else:
        raise ValueError('Method must be nearest or idw.')

    return prediction, supported


# Compare the two methods
def compare_methods(depth, train, val, test, radius=6):
    results = {}

    for method in ('nearest', 'idw'):
        results[method] = {}

        for stage, mask in [('validation', val), ('test', test)]:
            start = time.perf_counter()
            pred, supported = interpolate_depth(
                depth, train, mask, method, radius
            )
            runtime = time.perf_counter() - start

            used = mask & supported
            error = pred[used] - depth[used]

            results[method][stage] = {
                'requested': int(mask.sum()),
                'predicted': int(used.sum()),
                'coverage': float(used.sum() / mask.sum()) if mask.any() else 0,
                'rmse': float(np.sqrt(np.mean(error**2))) if error.size else None,
                'mae': float(np.mean(np.abs(error))) if error.size else None,
                'runtime_seconds': runtime
            }

    def score(method):
        m = results[method]['validation']
        return float('inf') if m['rmse'] is None else m['rmse'] / max(m['coverage'], 0.01)

    return {
        'selected_method': min(results, key=score),
        'results': results
    }


def compare_interpolators(depth, seed=42, radius=6):
    train, val, test = make_split(depth, seed)
    return compare_methods(depth, train, val, test, radius)


def compare_block_interpolators(depth, seed=42, radius=6):
    train, val, test = make_block_split(depth, seed=seed)
    return compare_methods(depth, train, val, test, radius)


print('Numerical methods ready.')


Numerical methods ready.


In [6]:

# Read dataset inventory
with zipfile.ZipFile(dataset_zip) as z:
    records = list(csv.DictReader(
        io.StringIO(z.read('inventory.csv').decode('utf-8-sig'))
    ))

# Group depth images by folder, season and tree
groups = {}

for entry in records:
    name = entry['name']

    if '_D-' not in name or not name.endswith('.png'):
        continue

    tree, season = name.split('_')[:2]

    if season in ('Summer', 'Winter'):
        key = (entry['source_folder'], season)
        groups.setdefault(key, {}).setdefault(tree, entry)

# Select three trees from each group
selection = []

for (folder, season), trees in sorted(groups.items()):
    for tree in sorted(trees)[:3]:
        selection.append((folder, season, trees[tree]))

print('Selected images:', len(selection))

for folder, season, entry in selection:
    print(folder, season, entry['name'])


Selected images: 12
Dataset Summer R01U01_Summer_D-12-40-11.png
Dataset Summer R01U02_Summer_D-12-40-26.png
Dataset Summer R01U03_Summer_D-12-40-37.png
Dataset Winter R01U01_Winter_D-11-23-20.png
Dataset Winter R01U02_Winter_D-11-23-42.png
Dataset Winter R01U03_Winter_D-11-23-45.png
Dataset (With Summer GT) Summer R01N01_Summer_D-12-40-50.png
Dataset (With Summer GT) Summer R01N02_Summer_D-12-40-59.png
Dataset (With Summer GT) Summer R01N03_Summer_D-12-41-4.png
Dataset (With Summer GT) Winter R01N01_Winter_D-11-23-49.png
Dataset (With Summer GT) Winter R01N02_Winter_D-11-23-52.png
Dataset (With Summer GT) Winter R01N03_Winter_D-11-23-55.png


In [7]:

# Download and evaluate the 12 depth images
out = []

for folder, season, entry in selection:

    # Download image from Google Drive
    data = service.files().get_media(
        fileId=entry['id'], supportsAllDrives=True
    ).execute()

    with Image.open(io.BytesIO(data)) as img:
        arr = np.array(img)

    # Convert RGB depth to grayscale if channels are identical
    if arr.ndim == 3:
        if not (np.array_equal(arr[:, :, 0], arr[:, :, 1]) and
                np.array_equal(arr[:, :, 1], arr[:, :, 2])):
            print('Skipped:', entry['name'])
            continue
        depth = arr[:, :, 0]
    else:
        depth = arr

    # Test both validation designs
    for design, fn in [
        ('random_pixels', compare_interpolators),
        ('spatial_blocks_12', compare_block_interpolators)
    ]:
        result = fn(depth, seed=42, radius=6)

        for method in ('nearest', 'idw'):
            m = result['results'][method]['test']

            out.append({
                'folder': folder,
                'season': season,
                'tree': entry['name'].split('_')[0],
                'image': entry['name'],
                'design': design,
                'method': method,
                'rmse': m['rmse'],
                'mae': m['mae'],
                'coverage': m['coverage'],
                'runtime_seconds': m['runtime_seconds'],
                'requested': m['requested'],
                'predicted': m['predicted']
            })

    print('Processed:', entry['name'])

print('Results:', len(out), 'rows')


Processed: R01U01_Summer_D-12-40-11.png
Processed: R01U02_Summer_D-12-40-26.png
Processed: R01U03_Summer_D-12-40-37.png
Processed: R01U01_Winter_D-11-23-20.png
Processed: R01U02_Winter_D-11-23-42.png
Processed: R01U03_Winter_D-11-23-45.png
Processed: R01N01_Summer_D-12-40-50.png
Processed: R01N02_Summer_D-12-40-59.png
Processed: R01N03_Summer_D-12-41-4.png
Processed: R01N01_Winter_D-11-23-49.png
Processed: R01N02_Winter_D-11-23-52.png
Processed: R01N03_Winter_D-11-23-55.png
Results: 48 rows


In [8]:
import pandas as pd
from google.colab import files

frame = pd.DataFrame(out)

frame.to_csv('MMA3001_appletree_depthmetric.csv', index=False)

print(frame.groupby(['design', 'method'])[
    ['rmse', 'mae', 'coverage', 'runtime_seconds']
].median())

files.download('MMA3001_appletree_depthmetric.csv')

FileNotFoundError: Cannot find file: MMA3001_appletree_depthmetric.csv